# Run Performance Benchmarks with GuideLLM

This notebook benchmarks your llm-d / MaaS model endpoint using [GuideLLM](https://github.com/vllm-project/guidellm) — the vLLM project's official benchmarking tool. Results feed into capacity planning (next notebook).

**What we'll do:**
1. Install GuideLLM
2. Discover the model endpoint URL
3. Run single-user latency test (synchronous)
4. Run maximum throughput test
5. Run a full sweep benchmark (10 strategies)
6. Parse and display results
7. Quick capacity estimate

## 1. Install GuideLLM

In [ ]:
!pip install -q guidellm pandas tabulate

import importlib
print(f"GuideLLM installed: {importlib.import_module('guidellm').__version__}")

## 2. Discover Model Endpoint

GuideLLM needs an OpenAI-compatible `/v1` endpoint. We'll use the MaaS gateway URL from earlier phases.

In [ ]:
import subprocess, json

cluster_domain = subprocess.run(
    ["oc", "get", "ingresses.config", "cluster", "-o", "jsonpath={.spec.domain}"],
    capture_output=True, text=True
).stdout.strip()

MODEL_URL = f"https://maas.{cluster_domain}"
MODEL_NAME = "Qwen/Qwen3-Coder-30B-A3B-Instruct-FP8"

print(f"Target endpoint: {MODEL_URL}/v1")
print(f"Model: {MODEL_NAME}")
print(f"\nVerifying model is available...")

result = subprocess.run(
    ["curl", "-sk", f"{MODEL_URL}/v1/models"],
    capture_output=True, text=True
)
try:
    models = json.loads(result.stdout)
    available = [m["id"] for m in models.get("data", [])]
    print(f"Available models: {available}")
    if MODEL_NAME in available:
        print(f"✅ {MODEL_NAME} is ready")
    else:
        print(f"⚠️  {MODEL_NAME} not found — check Phase 4 deployment")
except:
    print(f"⚠️  Could not parse model list — check gateway connectivity")

## 3. Single-User Latency Test (Synchronous)

Sends one request at a time — no concurrency. Measures baseline TTFT, ITL, and single-stream output tok/s. This represents the best-case developer experience.

In [ ]:
import subprocess
from pathlib import Path

RESULTS_DIR = Path("benchmark_results")
RESULTS_DIR.mkdir(exist_ok=True)

print("Running synchronous benchmark (single user, 512 prompt / 256 output tokens)...")
print("This takes ~60 seconds.\n")

cmd = f"""guidellm benchmark \
  --target "{MODEL_URL}/v1" \
  --model "{MODEL_NAME}" \
  --profile synchronous \
  --data "kind=synthetic_text,prompt_tokens=512,output_tokens=256" \
  --max-seconds 60 \
  --output-path "{RESULTS_DIR}/sync_results" \
  --backend-kwargs '{{"verify_ssl": false}}'"""

result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
print(result.stdout[-2000:] if result.stdout else "")
if result.returncode != 0:
    print(f"stderr: {result.stderr[-1000:]}")

## 4. Maximum Throughput Test

Sends as many concurrent requests as the server can handle. Finds the peak aggregate tok/s — the theoretical ceiling before requests start queuing.

In [ ]:
print("Running throughput benchmark (max concurrency, 512/256 tokens)...")
print("This takes ~60 seconds.\n")

cmd = f"""guidellm benchmark \
  --target "{MODEL_URL}/v1" \
  --model "{MODEL_NAME}" \
  --profile throughput \
  --data "kind=synthetic_text,prompt_tokens=512,output_tokens=256" \
  --max-seconds 60 \
  --output-path "{RESULTS_DIR}/throughput_results" \
  --backend-kwargs '{{"verify_ssl": false}}'"""

result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
print(result.stdout[-2000:] if result.stdout else "")
if result.returncode != 0:
    print(f"stderr: {result.stderr[-1000:]}")

## 5. Sweep Benchmark (10 Strategies)

The sweep profile runs all strategies in sequence: synchronous → throughput → 8 constant-rate steps. This produces the full latency-vs-load curve needed for capacity planning.

> **Duration:** ~10–12 minutes total (60s per strategy × 10 strategies).

In [ ]:
print("Running sweep benchmark (10 strategies, 512/256 tokens)...")
print("This takes ~10-12 minutes. Each strategy runs for 60 seconds.\n")

cmd = f"""guidellm benchmark \
  --target "{MODEL_URL}/v1" \
  --model "{MODEL_NAME}" \
  --profile sweep \
  --rate 10 \
  --data "kind=synthetic_text,prompt_tokens=512,output_tokens=256" \
  --max-seconds 60 \
  --output-path "{RESULTS_DIR}/sweep_results" \
  --backend-kwargs '{{"verify_ssl": false}}'"""

result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
print(result.stdout[-3000:] if result.stdout else "")
if result.returncode != 0:
    print(f"stderr: {result.stderr[-1000:]}")

## 6. Parse and Display Results

Load the benchmark output and display key metrics in a readable table.

In [ ]:
import json, glob
from pathlib import Path

def load_results(results_dir):
    """Find and load the most recent benchmark result file."""
    patterns = ["*.json", "**/*.json"]
    for pattern in patterns:
        files = sorted(glob.glob(str(results_dir / pattern)), key=lambda f: Path(f).stat().st_mtime, reverse=True)
        if files:
            with open(files[0]) as f:
                return json.load(f), files[0]
    return None, None

print("=== Benchmark Results Summary ===\n")

for name, subdir in [("Synchronous", "sync_results"), ("Throughput", "throughput_results"), ("Sweep", "sweep_results")]:
    data, filepath = load_results(RESULTS_DIR / subdir)
    if data:
        print(f"--- {name} ({filepath}) ---")
        if isinstance(data, dict):
            for key in ["ttft_ms", "itl_ms", "output_tokens_per_second", "requests_per_second"]:
                if key in data:
                    print(f"  {key}: {data[key]}")
        print()
    else:
        print(f"--- {name}: No results found (benchmark may not have completed) ---\n")

print("\nNote: If results are empty, check that GuideLLM completed without errors above.")
print("You can also view raw results in the benchmark_results/ directory.")

## 7. Quick Capacity Estimate

Using the 30% concurrency model: at any moment, ~30% of developers actively using the assistant are waiting on the model simultaneously.

In [ ]:
CONCURRENCY_RATIO = 0.30

print("=== Quick Capacity Estimate ===\n")
print("Enter your measured values (or use reference defaults):\n")

single_user_toks = float(input("Single-user output tok/s [default 93]: ") or "93")
peak_agg_toks = float(input("Peak aggregate output tok/s [default 1357]: ") or "1357")

theoretical_max = peak_agg_toks / (single_user_toks * CONCURRENCY_RATIO)
practical = theoretical_max * 0.6  # 60% of theoretical for SLO headroom

print(f"\n--- Results ---")
print(f"  Single-user tok/s:     {single_user_toks}")
print(f"  Peak aggregate tok/s:  {peak_agg_toks}")
print(f"  Concurrency ratio:     {CONCURRENCY_RATIO:.0%}")
print(f"  Theoretical max devs:  {theoretical_max:.0f}")
print(f"  Practical (with SLO):  {practical:.0f} developers per GPU replica")
print(f"\n  → For a team of {int(practical * 2)} developers, deploy 2 replicas.")

## Summary

| Benchmark | Key Metric | Your Result |
|-----------|-----------|-------------|
| Synchronous | Single-user output tok/s | (from Step 3) |
| Synchronous | TTFT baseline | (from Step 3) |
| Throughput | Peak aggregate tok/s | (from Step 4) |
| Sweep | Latency-vs-load curve | (from Step 5) |

**Reference baselines (Qwen3-Coder-30B FP8 on L40S):**
- Single-user: ~93 tok/s, 74ms TTFT, 10ms ITL
- Peak throughput: ~1,357 tok/s aggregate

→ Continue to `3_capacity_planning.ipynb` for detailed multi-replica projections and cost analysis.